# CENG 476 — Plant Disease Classification Using Deep Learning and Transfer Learning

**Student:** Emir EVREN — **ID:** 210444038

This notebook follows the technical requirements of the CENG 476 project specification. It records the final implementation, parameter values, experimental comparisons, and the justification for each major design decision. The aim is to document the methodology professionally and reproducibly rather than present a tutorial.

**Final audited benchmark:** Custom CNN 84.62%, ResNet18 97.66%, EfficientNet-B0 99.01%, 50/50 ensemble 99.14%.


## 1. Task, Dataset and Evaluation Protocol

The task is **single-label, 38-class image classification** on PlantVillage. Input tensors are RGB `3×224×224`; the output layer returns 38 raw logits.

Final ultra-strict split:
- train: **39,091**
- validation: **4,462**
- locked test: **10,709**
- total: **54,262**

A fixed **train / validation / test** protocol was used; **k-fold cross-validation was not used**. Train data updates parameters, validation data supports scheduler/checkpoint/hyperparameter/ensemble decisions, and the locked test is reserved for final evaluation.

K-fold CV was omitted because repeated full CNN fine-tuning is computationally expensive and a separate validation set plus locked test was already maintained. Reproducibility was instead checked with multiple random seeds.

Test labels and predictions were not used for training, tuning, checkpoint selection or ensemble-weight selection. Test images were used only in a deterministic model-independent integrity audit.


## 2. Preprocessing and Data Augmentation

### Training transform
- `RandomResizedCrop(224, scale=(0.80,1.00))`
- `RandomHorizontalFlip(p=0.5)`
- `RandomRotation(±15°)`
- `ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2)`
- `ToTensor()`
- ImageNet normalization: mean `[0.485,0.456,0.406]`, std `[0.229,0.224,0.225]`

Augmentation was applied only to training data. It introduces controlled variation in crop, orientation and color while preserving lesion information. Validation/test preprocessing is deterministic: `Resize(256) → CenterCrop(224) → ToTensor() → Normalize`.

ImageNet normalization is appropriate because ResNet18 and EfficientNet-B0 use ImageNet-pretrained weights.


In [ ]:
from torchvision import transforms
from torchvision.transforms import InterpolationMode
MEAN=[0.485,0.456,0.406]; STD=[0.229,0.224,0.225]
train_transform=transforms.Compose([
    transforms.RandomResizedCrop(224,scale=(0.80,1.00)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(15,interpolation=InterpolationMode.BILINEAR,fill=(128,128,128)),
    transforms.ColorJitter(brightness=0.2,contrast=0.2,saturation=0.2),
    transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
eval_transform=transforms.Compose([
    transforms.Resize(256),transforms.CenterCrop(224),
    transforms.ToTensor(),transforms.Normalize(MEAN,STD)])


## 3. Model Architecture

### Custom CNN
Four blocks of `Conv(3×3) → BatchNorm → ReLU → MaxPool(2×2)` with channels `32→64→128→256`, followed by adaptive average pooling, `Dropout(0.40)` and `Linear(256,38)`. Total trainable parameters: **399,142**.

Convolutions are suitable because disease classification depends on local spatial evidence such as texture, discoloration and lesion patterns. Pooling reduces spatial cost while increasing channel depth preserves feature capacity.

### ResNet18
ImageNet pretrained, full fine-tuning. Classifier: `Dropout(0.30) → Linear(512,38)`. Residual connections improve gradient flow. Parameters: **11,196,006**.

### EfficientNet-B0
ImageNet pretrained, full fine-tuning. Classifier: `Dropout(0.30) → Linear(1280,38)`. Parameters: **4,056,226**. It provided the best individual accuracy/parameter trade-off.


## 4. Batch Normalization, Dropout and Regularization

**Batch Normalization** is placed after each convolution and before ReLU in the custom CNN. It stabilizes intermediate activation scales and makes optimization more reliable. Training uses batch statistics; evaluation uses running statistics.

**Dropout** suppresses random activations during training and is disabled in evaluation. Baseline dropout is 0.40; transfer classifiers use 0.30.

Controlled baseline pilot:
| Dropout | Best Validation Macro-F1 |
|---:|---:|
| 0.20 | 0.4990 |
| 0.40 | 0.4953 |
| 0.60 | 0.4311 |

The 0.60 setting was too aggressive in the short pilot. Additional regularization consists of data augmentation and AdamW weight decay `1e-4`.


## 5. Activation Functions, Output and Loss

Custom CNN and ResNet18 use **ReLU**; EfficientNet-B0 retains its native **SiLU** activation. Nonlinear activations are required because a stack of linear transformations alone cannot represent complex nonlinear decision boundaries.

The output layer returns **38 raw logits**. Softmax is intentionally not placed before the loss.

The loss is unweighted `CrossEntropyLoss`:

\[
L=-\log\left(\frac{e^{z_y}}{\sum_j e^{z_j}}\right)
\]

CrossEntropyLoss expects raw logits and internally performs the stable LogSoftmax/NLL computation. Softmax is used only when probabilities are needed for ensemble voting, confidence, calibration or ROC analysis.


In [ ]:
import torch
from torch import nn
criterion=nn.CrossEntropyLoss()
print("Loss:",criterion.__class__.__name__)
print("Input: raw logits [batch,38]")
print("Softmax before loss: No")


Loss: CrossEntropyLoss
Input: raw logits [batch,38]
Softmax before loss: No


## 6. Optimization Strategy

All final models use **AdamW**, betas `(0.9,0.999)`, weight decay `1e-4`.

| Setting | Baseline | Transfer models |
|---|---:|---:|
| Max epochs | 15 | 12 |
| Batch size | 64 | 32 |
| Main/backbone LR | `5e-4` | `1e-4` |
| Classifier LR | — | `5e-4` |
| Dropout | 0.40 | 0.30 |
| Early-stop patience | 6 | 5 |

Baseline LR pilots included `1e-3`, `5e-4`, `3e-4`. The transfer backbone uses a smaller LR because pretrained representations should be updated conservatively; the newly initialized classifier uses the larger LR.

`ReduceLROnPlateau` monitors validation loss with factor 0.5, patience 2 and minimum LR `1e-6`. Validation loss is a smooth scheduler signal, whereas **validation Macro-F1** is used for checkpoint selection because it balances all classes.

Early stopping was implemented, but the selected final runs reached their configured maximum epoch count before it terminated training.


## 7. Training Loop and Sanity Check

Mini-batch order: `zero_grad → forward → loss → backward → optimizer step`. Gradients are cleared because PyTorch accumulates them by default. AMP was enabled on CUDA to improve memory efficiency and speed.

Before full training, a 38-image sanity test was run with augmentation off, dropout 0 and weight decay 0. The network reached **71.05% at step 30** and **100% at step 40**. This was an implementation check: the model, labels, loss, backpropagation and optimizer were capable of learning a tiny dataset.


## 8. Metrics

- **Accuracy:** fraction of correct predictions.
- **Precision:** `TP/(TP+FP)`.
- **Recall:** `TP/(TP+FN)`.
- **F1:** harmonic mean of precision and recall.
- **Macro-F1:** equal average of the 38 class-wise F1 scores; main checkpoint metric because class supports differ.
- **Weighted-F1:** F1 weighted by class support.
- **Confusion matrix:** identifies which classes are confused.
- **ROC-AUC:** one-vs-rest multiclass ranking metric.


## 9. Final Results

| Model | Accuracy | Macro-F1 | Weighted-F1 | Errors |
|---|---:|---:|---:|---:|
| Custom CNN | **84.62%** | 0.7813 | 0.8361 | 1,647 |
| ResNet18 | **97.66%** | 0.9686 | 0.9763 | 251 |
| EfficientNet-B0 | **99.01%** | 0.9874 | 0.9901 | 106 |
| 50/50 Ensemble | **99.14%** | 0.9897 | 0.9914 | 92 |

Transfer learning produced a large improvement over the scratch baseline. EfficientNet-B0 was the strongest individual model while using fewer parameters than ResNet18.


## 10. Ensemble and Overfitting Analysis

Soft voting uses `p_ens = 0.5 p_ResNet + 0.5 p_EfficientNet`. Candidate weights were selected **only on validation data**. The best validation Macro-F1 was the 50/50 setting: **0.992033**.

A deterministic clean-train subset (20 images/class = 760) was used for generalization analysis:

| Model | Clean train | Validation | Test |
|---|---:|---:|---:|
| Custom CNN | 77.89% | 83.55% | 84.62% |
| ResNet18 | 98.16% | 98.81% | 97.66% |
| EfficientNet-B0 | 99.74% | 98.68% | 99.01% |

EfficientNet's clean-train, validation and PlantVillage test scores are close, so severe conventional train-set overfitting is not supported.


## 11. Leakage Audit and Protocol Revision

The historical image-level protocol produced **99.76%** ensemble accuracy, which motivated an integrity audit.

Detected in the original protocol:
- exact cross-split duplicates: **10**
- perceptual near-duplicate pairs: **68**
- mapped same-physical-leaf cross-split groups: **4,956**

Different files can represent the same physical leaf, so an index-disjoint split is not necessarily specimen-independent.

The official PlantVillage split was adopted; 5 exact train/test collisions were removed from training. A final `dHash≤4` review found 39 strict cross-split pairs. With priority `test > validation > train`, **34 train + 4 validation + 0 test** images were quarantined.

Final split: **39,091 train / 4,462 validation / 10,709 test**.

No detected exact, mapped-leaf or strict `dHash≤4` overlap remains under the implemented audit. This is not a mathematical uniqueness guarantee for every unmapped specimen; leaf mapping covers about 75.7% of images.


## 12. Historical vs Final Results

| Model | Historical image-level | Final ultra-strict |
|---|---:|---:|
| Custom CNN | 87.42% | 84.62% |
| ResNet18 | 99.26% | 97.66% |
| EfficientNet-B0 | 99.52% | 99.01% |
| Ensemble | 99.76% | 99.14% |

The original evaluation was optimistic and contained documented overlap. The full accuracy difference is not attributed solely to leakage because protocol/training conditions also changed.


## 13. Additional Validation

**Seed stability:** seed 42 = 99.010%, seed 123 = 98.767%, seed 777 = 99.225%; mean **99.001%**, std **0.229 percentage points**. Seed 777 is not promoted as the benchmark because selecting the best test seed would be cherry-picking.

**Random-label control:** true-label validation accuracy after shuffled-label training = **1.84%**, near 38-class chance (**2.63%**). This is evidence against trivial pipeline/label leakage, not proof of zero leakage.

**Calibration / uncertainty:** EfficientNet ECE 0.003845, NLL 0.035704, Brier 0.016145. A 1,000-sample ordinary bootstrap gave approximately **98.81–99.20%** accuracy 95% CI.

**Robustness / Grad-CAM:** moderate brightness, contrast, JPEG and rotation changes were tolerated better than Gaussian blur and large occlusion. Grad-CAM was used as qualitative evidence, not causal proof.


## 14. External OOD Evaluation: PlantDoc

The final models were evaluated **without retraining** on 236 mapped Cropped-PlantDoc test images across 27 mapped source classes.

| Model | Accuracy | Mapped Macro-F1 |
|---|---:|---:|
| EfficientNet-B0 | **23.31%** | 0.2183 |
| Ensemble | **25.00%** | 0.2349 |

PlantDoc and PlantVillage differ in acquisition conditions and label ontology; this is an OOD stress test rather than a directly comparable benchmark.

The drop does not by itself prove training-image memorization. Since PlantVillage clean-train, validation and test performance are all high, the stronger interpretation is **strong within-domain generalization but weak cross-domain transfer / dataset dependence**.


## 15. Experimental Development Summary

| Change | Reason | Outcome |
|---|---|---|
| Custom CNN | scratch baseline | 84.62% |
| ResNet18 | evaluate transfer learning | 97.66% |
| EfficientNet-B0 | improve parameter efficiency | 99.01% |
| Dropout pilot | study regularization | 0.60 slowed learning |
| LR pilots + scheduler | improve convergence | final baseline 5e-4 |
| 50/50 ensemble | combine probabilities | 99.14%, 92 errors |
| Leakage audit | investigate 99.76% | duplicate/same-leaf overlap found |
| Ultra-strict split | strengthen evaluation | final ensemble 99.14% |
| Multi-seed training | test reproducibility | 99.001 ± 0.229 pp |
| PlantDoc OOD | test external validity | 23–25% |


## 16. Conclusion and Reproducibility

The custom CNN provides a transparent baseline; transfer learning provides the major performance gain. The final audited PlantVillage benchmark is **99.01% for EfficientNet-B0** and **99.14% for the validation-selected ensemble**.

The leakage audit showed that the original image-level protocol was not sufficiently conservative. Multi-seed analysis shows that near-99% PlantVillage performance is reproducible, while PlantDoc demonstrates a substantial external-validity limitation. The project therefore claims strong **controlled-domain** performance, not near-99% real-world field accuracy.

Reference scripts:
```bash
python src/train_baseline.py --epochs 15 --learning-rate 5e-4 --batch-size 64 --weight-decay 1e-4 --dropout 0.4
python src/train_resnet18_official.py
python src/train_efficientnet_official.py
```
Seed 42 is the reference run; 123 and 777 are stability runs. Configs, manifests, histories, checkpoints and audit outputs are stored under `outputs/`.
